# MediaPipe FaceMesh EAR Testing (Split Test)

Notebook ini mengevaluasi klasifikasi mata `open/close` berbasis EAR dari MediaPipe FaceMesh pada split test:

- `blinkblink-5/test/0 - close`
- `blinkblink-5/test/1 - open`

Laporan utama menggunakan `classification_report()`.

In [1]:
# Jalankan jika belum ada dependensi
# %pip install -q scikit-learn pandas tqdm

In [2]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

from implementation import EyeAnalyzer, SystemConfig

# Konfigurasi path test split
TEST_ROOT = Path("blinkblink-5") / "test"
CLASS_TO_LABEL = {
    "0 - close": 0,
    "1 - open": 1,
}

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if not TEST_ROOT.exists():
    raise FileNotFoundError(f"Path tidak ditemukan: {TEST_ROOT}")

print(f"Test root: {TEST_ROOT.resolve()}")

Test root: E:\Ghozy\Tel-U\AI Lab\blinkblink\eye-blink-decoder\blinkblink-5\test


In [3]:
# Kumpulkan daftar file test
samples = []
for class_name, label in CLASS_TO_LABEL.items():
    class_dir = TEST_ROOT / class_name
    if not class_dir.exists():
        print(f"Warning: folder tidak ada -> {class_dir}")
        continue

    image_files = [p for p in class_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTS]
    for p in image_files:
        samples.append({"path": p, "label": label, "class_name": class_name})

if not samples:
    raise RuntimeError("Tidak ada gambar yang ditemukan pada test split.")

print(f"Total sample ditemukan: {len(samples)}")
df = pd.DataFrame(samples)
display(df.head())

Total sample ditemukan: 429


,path,label,class_name
0,blinkblink-5\test\0 - close\frame_000002_jpg.r...,0,0 - close
1,blinkblink-5\test\0 - close\frame_000002_jpg.r...,0,0 - close
2,blinkblink-5\test\0 - close\frame_000009_jpg.r...,0,0 - close
3,blinkblink-5\test\0 - close\frame_000009_jpg.r...,0,0 - close
4,blinkblink-5\test\0 - close\frame_000010_jpg.r...,0,0 - close


In [4]:
# Inferensi EAR dengan MediaPipe FaceMesh
config = SystemConfig()
analyzer = EyeAnalyzer()

rows = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating"):
    img_path = str(row["path"])
    img = cv2.imread(img_path)

    if img is None:
        rows.append({
            "path": img_path,
            "label": int(row["label"]),
            "ear": np.nan,
            "detected": False,
        })
        continue

    eye_data, _ = analyzer.process_frame(img, config)
    rows.append({
        "path": img_path,
        "label": int(row["label"]),
        "ear": float(eye_data.avg_ear) if eye_data.landmarks_detected else np.nan,
        "detected": bool(eye_data.landmarks_detected),
    })

# Pastikan resource dilepas
analyzer.close()

res_df = pd.DataFrame(rows)
print("Ringkasan deteksi FaceMesh:")
print(res_df["detected"].value_counts(dropna=False))

valid_df = res_df[res_df["detected"]].copy()
if valid_df.empty:
    raise RuntimeError("Tidak ada sample dengan landmark terdeteksi.")

# Threshold EAR (midpoint mean per class)
close_mean = valid_df.loc[valid_df["label"] == 0, "ear"].mean()
open_mean = valid_df.loc[valid_df["label"] == 1, "ear"].mean()

if np.isnan(close_mean) or np.isnan(open_mean):
    # Fallback aman jika salah satu kelas gagal terdeteksi
    threshold = float(valid_df["ear"].median())
else:
    threshold = float((close_mean + open_mean) / 2.0)

valid_df["pred"] = (valid_df["ear"] >= threshold).astype(int)

print(f"EAR threshold: {threshold:.6f}")
print(f"Close mean EAR: {close_mean:.6f}")
print(f"Open  mean EAR: {open_mean:.6f}")

Evaluating:   0%|          | 0/429 [00:00<?, ?it/s]

Ringkasan deteksi FaceMesh:
detected
True    429
Name: count, dtype: int64
EAR threshold: 0.119968
Close mean EAR: 0.075323
Open  mean EAR: 0.164614


In [5]:
# Evaluasi klasifikasi
y_true = valid_df["label"].tolist()
y_pred = valid_df["pred"].tolist()

print("classification_report():")
print(classification_report(y_true, y_pred, labels=[0, 1], target_names=["close", "open"], digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(cm, index=["true_close", "true_open"], columns=["pred_close", "pred_open"])

print("Confusion Matrix:")
display(cm_df)

print("\nContoh prediksi:")
display(valid_df[["path", "label", "ear", "pred"]].head(20))

classification_report():
              precision    recall  f1-score   support

       close     0.4384    0.8649    0.5818        37
        open     0.9860    0.8954    0.9385       392

    accuracy                         0.8928       429
   macro avg     0.7122    0.8801    0.7602       429
weighted avg     0.9387    0.8928    0.9077       429

Confusion Matrix:


,pred_close,pred_open
true_close,32,5
true_open,41,351



Contoh prediksi:


,path,label,ear,pred
0,blinkblink-5\test\0 - close\frame_000002_jpg.r...,0,0.031439,0
1,blinkblink-5\test\0 - close\frame_000002_jpg.r...,0,0.089344,0
2,blinkblink-5\test\0 - close\frame_000009_jpg.r...,0,0.124829,1
3,blinkblink-5\test\0 - close\frame_000009_jpg.r...,0,0.085245,0
4,blinkblink-5\test\0 - close\frame_000010_jpg.r...,0,0.026974,0
5,blinkblink-5\test\0 - close\frame_000013_jpg.r...,0,0.100837,0
6,blinkblink-5\test\0 - close\frame_000018_jpg.r...,0,0.080958,0
7,blinkblink-5\test\0 - close\frame_000021_jpg.r...,0,0.086887,0
8,blinkblink-5\test\0 - close\frame_000029_jpg.r...,0,0.032487,0
9,blinkblink-5\test\0 - close\frame_000029_jpg.r...,0,0.070894,0
